# LACE-VQA — remaining runs (clean notebook)
Runtime: **T4 GPU**. Run cells **top to bottom**, one at a time; wait for each to finish.
If the session disconnects: rerun cells 1–3, then the step you were on (finished results are skipped).

## Setup (every session)

In [ ]:
# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/access-vqa'

In [ ]:
# 2. Unpack code + install (should end with: Tesla T4)
!cd /content && rm -rf access-vqa && unzip -q -o $DRIVE/access-vqa.zip -d /content && ls /content/access-vqa
%cd /content/access-vqa
!pip install -q -r requirements.txt
import torch; print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
!grep -c 'max_side 640' scripts/run_all.sh   # should print 4

In [ ]:
# 3. Restore results from Drive, annotations, auto-backup (should end with: auto-backup started)
%env DATA=/content/data
%env WORK=/content/work
%env WORKERS=2
%env ZIP_CACHE=/content/drive/MyDrive/access-vqa/zips
!mkdir -p /content/work && rsync -a --exclude 'feats/' /content/drive/MyDrive/access-vqa/work/ /content/work/ 2>/dev/null; ls /content/work
!mkdir -p /content/data/vizwiz/Annotations && unzip -q -o /content/drive/MyDrive/access-vqa/zips/Annotations.zip -d /content/data/vizwiz/Annotations && ls /content/data/vizwiz/Annotations

def backup():
    !rsync -a /content/work/ /content/drive/MyDrive/access-vqa/work/

import subprocess
if subprocess.run(["pgrep", "-f", "autobackup-loop"], capture_output=True).returncode != 0:
    subprocess.Popen(["bash", "-c", "while true; do sleep 600; rsync -a /content/work/ /content/drive/MyDrive/access-vqa/work/; done # autobackup-loop"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("auto-backup started (every 10 min)")
else:
    print("auto-backup already running")

## Experiments

In [ ]:
# 4. Validation images (~5 min) — should print 7750
!bash scripts/download_data.sh vizwiz_val
!ls /content/data/vizwiz/val | wc -l

In [ ]:
# 5. INT8 accuracy (~20-30 min) — skips the 4 done, runs 2 new
!STAGES="int8acc" bash scripts/run_all.sh 2>&1 | grep -E '^\{"model|skip|Error'
backup()

In [ ]:
# 6. Speed benchmark (~30-45 min)
!BENCH_ARGS="--iters 30" STAGES="benchmark" bash scripts/run_all.sh 2>&1 | grep -E '^(viz_|vilt|blip)|Error'
!ls -la /content/work/results/benchmark.json
backup()

In [ ]:
# 7. Robustness (~1.5-2.5 h) — 125 lines in total; progress check: cell 7b
!STAGES="robustness" bash scripts/run_all.sh 2>&1 | grep -E '^\{"model|Error'
backup()

In [ ]:
# 7b. (optional, run anytime in another tab/after) how many robustness results exist (target 125)
!ls /content/work/results/rob_*.json 2>/dev/null | wc -l

## Final

In [ ]:
# 8. Tables + bundle -> saved to Drive as results_bundle.zip (send this file)
!STAGES="tables" bash scripts/run_all.sh
!cd /content && rm -f results_bundle.zip && zip -qr results_bundle.zip access-vqa/paper work/runs/*/results.json work/runs/*/log.jsonl work/results
!cp /content/results_bundle.zip /content/drive/MyDrive/access-vqa/
!ls -la /content/drive/MyDrive/access-vqa/results_bundle.zip